# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` Python library and pandas. All schema and field references follow the `@id` conventions, in line with Croissant best practices.

### Dataset Source
The dataset is described and structured via a Croissant JSON-LD schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. All entities are referenced by their `@id` fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore available record sets and their fields. Each entity is referenced by its `@id` field for strict reproducibility and schema navigation.

In [ ]:
# List all record sets by `@id`
print("Available record sets (`@id`s):")
record_sets = []
for rs in dataset.metadata.record_sets:
    print(f"- {rs.id} (name: {getattr(rs, 'name', '[unnamed]')})")
    record_sets.append(rs.id)
    
    # List fields for each record set
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.id}: {getattr(field, 'name', '[unnamed]')}, dataType: {getattr(field, 'data_type', '[unknown]')}")
print("\nTotal record sets found:", len(record_sets))

## 3. Data Extraction
Load records for each record set into pandas DataFrames. Columns correspond to field `@id`s and all access is by `@id` by convention.

In [ ]:
# Extract data from each record set using their `@id`
dataframes = {}
for record_set_id in record_sets:
    # Get all records as dicts
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set '{record_set_id}' with shape {df.shape}")
    # Show the columns (field @ids)
    print("Columns (@id):", df.columns.tolist())
    print(df.head(2))
    print()
# Example: Preview the columns of the primary patient records record set
if record_sets:
    example_record_set_id = record_sets[0]
    print(f"Primary record set columns (@id): {dataframes[example_record_set_id].columns.tolist()}")
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's demonstrate basic filtering, normalization, and one-hot encoding/grouping, using numeric and categorical fields referenced strictly by their `@id` values.

Select one record set and two field `@id`s below (numeric and grouping). These can be adjusted after viewing the column names above.

In [ ]:
# Choose record set and relevant field @ids (adjust as per record set columns from above)

# 1. Set (manually or via logic) the record set @id to analyze
record_set_id = record_sets[0]  # Replace with correct @id if needed
df = dataframes[record_set_id]

# 2. List candidate numeric and group fields from columns
print("Available fields (`@id`s):", df.columns.tolist())

# Suppose we picked a numeric and group field by @id
# Example guess (update if known from the previous exploration):
numeric_field_id = None
group_field_id = None

# Attempt to infer column (@id) for numeric field
for c in df.columns:
    # Look for typical numeric clinical field names
    if any(s in c.lower() for s in ['age', 'interval', 'duration', 'years']):
        numeric_field_id = c
    if any(s in c.lower() for s in ['sex', 'gender', 'anatomical', 'location', 'msi', 'group', 'status']):
        group_field_id = c
if numeric_field_id is None:
    numeric_field_id = df.columns[0]  # fallback
if group_field_id is None:
    group_field_id = df.columns[1] if len(df.columns) > 1 else df.columns[0]

print(f"Numeric field @id chosen: {numeric_field_id}")
print(f"Group field @id chosen: {group_field_id}")

# Attempt conversion to numeric (silently drop if fails)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter: Show cases where the numeric value > threshold
threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id]).any() else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered (where {numeric_field_id} > {threshold}): {len(filtered_df)} records\n", filtered_df.head())

# Normalize the numeric field
if len(filtered_df) > 0:
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group and aggregate
if group_field_id in df.columns:
    grouped = filtered_df.groupby(group_field_id, dropna=False)[numeric_field_id].agg(['mean','count'])
    print(f"Grouped statistics by {group_field_id}:")
    print(grouped.head())

## 5. Visualization
Visualize numeric and categorical relationships, referencing fields by their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Distribution of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=12, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Grouped numeric field by group
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load Croissant metadata and records with `mlcroissant` using only `@id` references
- Examine available record sets and their field structure
- Ingest record sets into pandas for EDA
- Filter, normalize, group, and visualize the data by field `@id`

This approach ensures reproducibility and schema-robustness for downstream ML workflows. Explore further using the exact field and record set `@id`s discovered in Section 2.